# Notebook 07 — Post-lock development analyses (revision)

**Purpose.** Reproduce the analyses added after review, all on the train and tune partitions only (the locked test is never read): the three-seed mechanism ladder and pretraining/width controls under the frozen 3-epoch schedule, the byte-budget sweep (uniform vs foveated at 8/16/32 local bins), the Dirichlet–multinomial vote head (CAPE-EEG v2), the patient-count learning curve, and inference latency/energy measurements. **Outputs:** `results/aggregate/table3b_*.csv`, `table6_*.csv`, `inference_measurements.json`, `paper/numbers*.tex`, `paper/figures/fig_*.pdf`. **GPU:** development pool; every run is ledgered. Runs that already exist with matching hashes are reused, never retrained.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


repo: $CAPE_ROOT/cape-eeg
workspace root: $CAPE_ROOT
data root: $CAPE_ROOT
private: $CAPE_ROOT/private


## Three-seed ladder and controls, byte-budget sweep, Dirichlet–multinomial head (frozen 3-epoch schedule)

In [2]:
import subprocess
for seed in ['101', '202', '303']:
    for cfgs, stage in [('B2 A1 A2 P B3 B3S B3H', None), ('B2_t16 A1_t16 B2_t8 A1_t8 P_DM', None)]:
        env = dict(os.environ, CONFIGS=cfgs, EPOCHS='3', TAG='ep3', SEED=seed, SKIP_CPU_BASELINES='1')
        p = subprocess.run(['bash', str(REPO / 'scripts' / 'run_dev_pipeline.sh')], capture_output=True, text=True, env=env)
        print('\n'.join(l for l in p.stdout.splitlines() if l.startswith('run ') or 'already PASS' in l or 'status:' in l)); assert p.returncode == 0

run dev_B2_s101_4455edaa_2406ce59_2743aee0
status: PASS  | best: {'tune_patient_kl': 0.968485123059124, 'epoch': 2} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.34, 'sys_available_min_gib': 111.46}
run dev_A1_s101_4455edaa_2406ce59_1cb15135
status: PASS  | best: {'tune_patient_kl': 0.968725152680912, 'epoch': 2} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.38, 'sys_available_min_gib': 111.39}
run dev_A2_s101_4455edaa_2406ce59_370955f8
status: PASS  | best: {'tune_patient_kl': 1.0111372377591552, 'epoch': 2} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.39, 'sys_available_min_gib': 111.31}
run dev_P_s101_4455edaa_2406ce59_57d7d79e
status: PASS  | best: {'tune_patient_kl': 0.9890071443701122, 'epoch': 3} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.61, 'sys_available_min_gib': 111.16}
run dev_B3_s101_4455edaa_2406ce59_2d2587fc
status: PASS  | best: {'tune_patient_kl': 0.86778488

run dev_B2_t16_s101_4455edaa_2406ce59_ebc0be1c already PASS; reuse (hashes match)
run dev_A1_t16_s101_4455edaa_2406ce59_6ab8401f already PASS; reuse (hashes match)
run dev_B2_t8_s101_4455edaa_2406ce59_8277147a already PASS; reuse (hashes match)
run dev_A1_t8_s101_4455edaa_2406ce59_53d850f7 already PASS; reuse (hashes match)
run dev_P_DM_s101_4455edaa_2406ce59_a5a05314 already PASS; reuse (hashes match)


run dev_B2_s202_4455edaa_2406ce59_2743aee0
status: PASS  | best: {'tune_patient_kl': 0.9640911378732939, 'epoch': 2} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.33, 'sys_available_min_gib': 111.44}
run dev_A1_s202_4455edaa_2406ce59_1cb15135
status: PASS  | best: {'tune_patient_kl': 0.9735310068052317, 'epoch': 3} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.33, 'sys_available_min_gib': 111.41}
run dev_A2_s202_4455edaa_2406ce59_370955f8
status: PASS  | best: {'tune_patient_kl': 0.9622607226837937, 'epoch': 1} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.38, 'sys_available_min_gib': 111.49}
run dev_P_s202_4455edaa_2406ce59_57d7d79e
status: PASS  | best: {'tune_patient_kl': 1.0281197480617819, 'epoch': 3} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.57, 'sys_available_min_gib': 111.27}
run dev_B3_s202_4455edaa_2406ce59_2d2587fc
status: PASS  | best: {'tune_patient_kl': 0.866623

run dev_B2_t16_s202_4455edaa_2406ce59_ebc0be1c already PASS; reuse (hashes match)
run dev_A1_t16_s202_4455edaa_2406ce59_6ab8401f already PASS; reuse (hashes match)
run dev_B2_t8_s202_4455edaa_2406ce59_8277147a already PASS; reuse (hashes match)
run dev_A1_t8_s202_4455edaa_2406ce59_53d850f7 already PASS; reuse (hashes match)
run dev_P_DM_s202_4455edaa_2406ce59_a5a05314 already PASS; reuse (hashes match)


run dev_B2_s303_4455edaa_2406ce59_2743aee0
status: PASS  | best: {'tune_patient_kl': 0.9992122264594441, 'epoch': 3} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.37, 'sys_available_min_gib': 111.43}
run dev_A1_s303_4455edaa_2406ce59_1cb15135
status: PASS  | best: {'tune_patient_kl': 0.9918045312798502, 'epoch': 2} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.33, 'sys_available_min_gib': 111.51}
run dev_A2_s303_4455edaa_2406ce59_370955f8
status: PASS  | best: {'tune_patient_kl': 0.9822042073638847, 'epoch': 1} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.38, 'sys_available_min_gib': 111.48}
run dev_P_s303_4455edaa_2406ce59_57d7d79e
status: PASS  | best: {'tune_patient_kl': 0.9391419635361047, 'epoch': 1} | peak: {'cuda_alloc_gib': 0.44, 'cuda_reserved_gib': 0.61, 'rss_gib': 7.56, 'sys_available_min_gib': 111.32}
run dev_B3_s303_4455edaa_2406ce59_2d2587fc
status: PASS  | best: {'tune_patient_kl': 0.883386

run dev_B2_t16_s303_4455edaa_2406ce59_ebc0be1c already PASS; reuse (hashes match)
run dev_A1_t16_s303_4455edaa_2406ce59_6ab8401f already PASS; reuse (hashes match)
run dev_B2_t8_s303_4455edaa_2406ce59_8277147a already PASS; reuse (hashes match)
run dev_A1_t8_s303_4455edaa_2406ce59_53d850f7 already PASS; reuse (hashes match)
run dev_P_DM_s303_4455edaa_2406ce59_a5a05314 already PASS; reuse (hashes match)


## Patient-count learning curve (P and B3 on 25/50/75% of training patients)

In [3]:
for seed in ['101', '202', '303']:
    for frac in ['0.25', '0.5', '0.75']:
        env = dict(os.environ, CONFIGS='P B3', EPOCHS='3', TAG=f'lc{frac}', SEED=seed, PFRAC=frac, STAGE='learning_curve', SKIP_CPU_BASELINES='1')
        p = subprocess.run(['bash', str(REPO / 'scripts' / 'run_dev_pipeline.sh')], capture_output=True, text=True, env=env)
        print('\n'.join(l for l in p.stdout.splitlines() if 'already PASS' in l or 'status:' in l)); assert p.returncode == 0

run dev_P_s101_4455edaa_2406ce59_6eaad105 already PASS; reuse (hashes match)
run dev_B3_s101_4455edaa_2406ce59_8530254c already PASS; reuse (hashes match)


run dev_P_s101_4455edaa_2406ce59_dfb0a2b5 already PASS; reuse (hashes match)
run dev_B3_s101_4455edaa_2406ce59_70bdfc8d already PASS; reuse (hashes match)


run dev_P_s101_4455edaa_2406ce59_715e786b already PASS; reuse (hashes match)
run dev_B3_s101_4455edaa_2406ce59_37a970ef already PASS; reuse (hashes match)


run dev_P_s202_4455edaa_2406ce59_6eaad105 already PASS; reuse (hashes match)
run dev_B3_s202_4455edaa_2406ce59_8530254c already PASS; reuse (hashes match)


run dev_P_s202_4455edaa_2406ce59_dfb0a2b5 already PASS; reuse (hashes match)
run dev_B3_s202_4455edaa_2406ce59_70bdfc8d already PASS; reuse (hashes match)


run dev_P_s202_4455edaa_2406ce59_715e786b already PASS; reuse (hashes match)
run dev_B3_s202_4455edaa_2406ce59_37a970ef already PASS; reuse (hashes match)


run dev_P_s303_4455edaa_2406ce59_6eaad105 already PASS; reuse (hashes match)
run dev_B3_s303_4455edaa_2406ce59_8530254c already PASS; reuse (hashes match)


run dev_P_s303_4455edaa_2406ce59_dfb0a2b5 already PASS; reuse (hashes match)
run dev_B3_s303_4455edaa_2406ce59_70bdfc8d already PASS; reuse (hashes match)


run dev_P_s303_4455edaa_2406ce59_715e786b already PASS; reuse (hashes match)
run dev_B3_s303_4455edaa_2406ce59_37a970ef already PASS; reuse (hashes match)


## Inference latency and energy (idle GPU required)

In [4]:
if not (REPO / 'results' / 'aggregate' / 'inference_measurements.json').exists():
    run(['measure_inference.py'])
im = read_json(REPO / 'results' / 'aggregate' / 'inference_measurements.json'); print('boundary:', im['boundary'][:160], '...')
for cid in ['P', 'B3']:
    r = im[cid]; print(cid, 'GPU b32 %.2f ms, %.0f win/s, %.1f W, %.3f mJ/window incremental | CPU 1 thread %.1f ms' % (r['gpu_batch_32']['median_ms'], r['gpu_batch_32']['windows_per_s'], r['gpu_batch_32']['mean_power_w'], r['gpu_batch_32']['energy_mj_per_window_incremental'], r['cpu_threads_1_batch_1']['median_ms']))

boundary: GPU package power as reported by nvidia-smi power.draw on the GB10 unified-memory system, sampled at 10 Hz during a 20 s sustained inference loop; idle baseline ...
P GPU b32 2.71 ms, 9883 win/s, 41.4 W, 2.941 mJ/window incremental | CPU 1 thread 8.8 ms
B3 GPU b32 1.86 ms, 15262 win/s, 46.9 W, 2.269 mJ/window incremental | CPU 1 thread 10.8 ms


## Tables and paper figures

In [5]:
p = subprocess.run(['bash', str(REPO / 'paper' / 'prepare_v2.sh')], capture_output=True, text=True, env=os.environ)
print('\n'.join(l for l in (p.stdout + p.stderr).splitlines() if l.strip() and not any(w in l for w in ('Warning', 'warn', 'Requested font'))))
import pandas as pd
for t in ['table3b_multiseed_ladder', 'table6_byte_sweep_paired', 'table6_learning_curve', 'table6_dirichlet_multinomial']:
    print('\n==', t); print(pd.read_csv(REPO / 'results' / 'aggregate' / f'{t}.csv').round(4).to_string(index=False))

dev_A2_s101_4455edaa_2406ce59_370955f8 tune full: 10667 rows in 1.7s -> tune_full_last.parquet; finite=True
dev_A2_s202_4455edaa_2406ce59_370955f8 tune full: 10667 rows in 1.7s -> tune_full_last.parquet; finite=True
dev_A2_s303_4455edaa_2406ce59_370955f8 tune full: 10667 rows in 1.8s -> tune_full_last.parquet; finite=True
dev_P_s101_4455edaa_2406ce59_57d7d79e tune full: 10667 rows in 1.7s -> tune_full_last.parquet; finite=True
dev_P_s202_4455edaa_2406ce59_57d7d79e tune full: 10667 rows in 1.8s -> tune_full_last.parquet; finite=True
dev_P_s303_4455edaa_2406ce59_57d7d79e tune full: 10667 rows in 1.7s -> tune_full_last.parquet; finite=True
encoding  time_bins  kl_mean  kl_sd  n  local_bytes
foveated          8   0.9989 0.0337  3         4352
 uniform          8   1.0136 0.0330  3         4352
foveated         16   1.0067 0.0211  3         8704
 uniform         16   1.0039 0.0171  3         8704
foveated         32   0.9795 0.0248  6        17408
 uniform         32   0.9802 0.0142  6     